# Activity & Readiness Analysis

Explore daily activity (steps, calories, MET minutes) and readiness scores with contributor breakdowns.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from oura_client import OuraClient

client = OuraClient(sandbox=True)

START = "2025-01-01"
END = "2025-03-01"

## Fetch Activity Data

In [ ]:
activities = client.get_daily_activity(start_date=START, end_date=END)
print(f"Fetched {len(activities)} activity records")

act_df = pd.DataFrame([
    {
        "day": a.day,
        "score": a.score,
        "steps": a.steps,
        "active_cal": a.active_calories,
        "total_cal": a.total_calories,
        "high_min": a.high_activity_minutes,
        "sedentary_h": a.sedentary_hours,
        "eq_walk_km": a.equivalent_walking_distance / 1000,
    }
    for a in activities
]).sort_values("day").reset_index(drop=True)

act_df.head(10)

## Steps & Active Calories Trend

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=act_df["day"], y=act_df["steps"], name="Steps", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=act_df["day"], y=act_df["active_cal"], name="Active Cal", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Daily Steps & Active Calories", template="plotly_white")
fig.update_yaxes(title_text="Steps", secondary_y=False)
fig.update_yaxes(title_text="Active Calories (kcal)", secondary_y=True)
fig.show()

## Fetch Readiness Data

In [ ]:
readiness_list = client.get_daily_readiness(start_date=START, end_date=END)
print(f"Fetched {len(readiness_list)} readiness records")

rd_df = pd.DataFrame([
    {
        "day": r.day,
        "score": r.score,
        "temp_dev": r.temperature_deviation,
        "hrv_balance": r.contributors.hrv_balance,
        "body_temp": r.contributors.body_temperature,
        "prev_night": r.contributors.previous_night,
        "sleep_balance": r.contributors.sleep_balance,
        "recovery_index": r.contributors.recovery_index,
        "resting_hr": r.contributors.resting_heart_rate,
        "activity_balance": r.contributors.activity_balance,
    }
    for r in readiness_list
]).sort_values("day").reset_index(drop=True)

rd_df.head(10)

## Readiness Score with Contributors

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=rd_df["day"], y=rd_df["score"], name="Readiness Score", mode="lines+markers", line=dict(width=3)))

contributor_cols = ["hrv_balance", "body_temp", "prev_night", "sleep_balance", "recovery_index", "resting_hr"]
for col in contributor_cols:
    fig.add_trace(go.Scatter(x=rd_df["day"], y=rd_df[col], name=col.replace("_", " ").title(), mode="lines", opacity=0.5))

fig.update_layout(title="Readiness Score & Contributors", yaxis_title="Score (0-100)", template="plotly_white")
fig.show()

## Activity Score vs Readiness Score

In [ ]:
merged = act_df.merge(rd_df, on="day", suffixes=("_act", "_rdy"))

fig = px.scatter(
    merged, x="score_act", y="score_rdy",
    size="steps", hover_data=["day"],
    title="Activity Score vs Readiness Score",
    labels={"score_act": "Activity Score", "score_rdy": "Readiness Score"},
    template="plotly_white",
    trendline="ols",
)
fig.show()

## Readiness Contributor Heatmap

Which contributors are dragging your readiness score down most often?

In [ ]:
contrib_data = rd_df.set_index("day")[contributor_cols].dropna()
if not contrib_data.empty:
    fig = px.imshow(
        contrib_data.T,
        aspect="auto",
        title="Readiness Contributors Over Time",
        labels={"x": "Date", "y": "Contributor", "color": "Score"},
        color_continuous_scale="RdYlGn",
    )
    fig.show()
else:
    print("No contributor data available for heatmap.")